
# Financial Risk Data Quality & Portfolio Monitoring Analytics Lab

## Module 04: Python-Based Data Quality Automation

### Business Objective
Develop a Python-based data quality validation process for a synthetic loan portfolio.

The objective is to identify, classify, and report data quality exceptions before portfolio information is used for risk reporting and management analysis.

### Key Controls
1. Mandatory field validation
2. Duplicate record detection
3. Referential integrity checks
4. Financial and DPD logic validation
5. Reporting period validation
6. SQL-to-Python reconciliation

### Technology Stack
- Python
- Pandas
- NumPy
- Jupyter Notebook
- PostgreSQL

### Expected Output
A structured data quality exception report containing control name, affected records, exception count, severity, and validation status.


In [1]:
# ============================================================
# STEP 2: PYTHON ENVIRONMENT VALIDATION
# ============================================================

import os
import sys
import pandas as pd
import numpy as np

print("PYTHON ENVIRONMENT CHECK")
print("-" * 45)

print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

print("\nCURRENT WORKING DIRECTORY")
print("-" * 45)

print(os.getcwd())

print("\nFILES AVAILABLE IN CURRENT DIRECTORY")
print("-" * 45)

for item in os.listdir():
    print(item)

print("\nENVIRONMENT VALIDATION: PASSED")

PYTHON ENVIRONMENT CHECK
---------------------------------------------
Python version: 3.13.9
Pandas version: 2.3.3
NumPy version: 2.3.5

CURRENT WORKING DIRECTORY
---------------------------------------------
C:\DATA ANALYSIS\financial-risk-portfolio-analytics\python

FILES AVAILABLE IN CURRENT DIRECTORY
---------------------------------------------
.ipynb_checkpoints
01_synthetic_loan_portfolio_generation.ipynb
04_data_quality_automation.ipynb

ENVIRONMENT VALIDATION: PASSED


In [2]:
from pathlib import Path

# Identify the main project directory
project_folder = Path.cwd().parent

print("PROJECT DIRECTORY")
print(project_folder)

print("\nPROJECT FOLDER CONTENTS")
print("-" * 50)

for item in sorted(project_folder.iterdir()):
    if item.is_dir():
        print(f"[FOLDER] {item.name}")
    else:
        print(f"[FILE]   {item.name}")

PROJECT DIRECTORY
C:\DATA ANALYSIS\financial-risk-portfolio-analytics

PROJECT FOLDER CONTENTS
--------------------------------------------------
[FOLDER] data
[FOLDER] docs
[FOLDER] python
[FOLDER] reports
[FOLDER] sql


In [3]:
# ============================================================
# STEP 4: DATA FOLDER INSPECTION
# ============================================================

data_folder = project_folder / "data"

print("DATA DIRECTORY")
print(data_folder)

print("\nDATA FOLDER STRUCTURE")
print("-" * 50)

for item in sorted(data_folder.rglob("*")):
    if item.is_file():
        print(f"[FILE] {item.relative_to(project_folder)}")
    elif item.is_dir():
        print(f"[FOLDER] {item.relative_to(project_folder)}")

DATA DIRECTORY
C:\DATA ANALYSIS\financial-risk-portfolio-analytics\data

DATA FOLDER STRUCTURE
--------------------------------------------------
[FOLDER] data\processed
[FILE] data\processed\account_summary.csv
[FILE] data\processed\customer_details.csv
[FILE] data\processed\loan_accounts.csv
[FILE] data\processed\payment_records.csv
[FILE] data\processed\portfolio_snapshots.csv
[FILE] data\processed\product_reference.csv
[FILE] data\processed\repayment_schedule.csv
[FOLDER] data\raw
[FILE] data\raw\customer_details.csv
[FILE] data\raw\loan_accounts.csv


In [4]:
# ============================================================
# STEP 5: LOAD SYNTHETIC PORTFOLIO DATASETS
# ============================================================

processed_folder = data_folder / "processed"

datasets = {
    "customer_details": "customer_details.csv",
    "loan_accounts": "loan_accounts.csv",
    "product_reference": "product_reference.csv",
    "repayment_schedule": "repayment_schedule.csv",
    "payment_records": "payment_records.csv",
    "account_summary": "account_summary.csv",
    "portfolio_snapshots": "portfolio_snapshots.csv"
}

portfolio_data = {}

for table_name, file_name in datasets.items():

    file_path = processed_folder / file_name

    portfolio_data[table_name] = pd.read_csv(file_path)

    print(f"{table_name}: Loaded successfully")

print("\nAll seven datasets loaded.")

customer_details: Loaded successfully
loan_accounts: Loaded successfully
product_reference: Loaded successfully
repayment_schedule: Loaded successfully
payment_records: Loaded successfully
account_summary: Loaded successfully
portfolio_snapshots: Loaded successfully

All seven datasets loaded.


In [5]:
# ============================================================
# MILESTONE 1: COMPLETE DATASET PROFILING
# ============================================================

profile_results = []

for table_name, df in portfolio_data.items():

    profile_results.append({
        "table_name": table_name,
        "total_rows": df.shape[0],
        "total_columns": df.shape[1],
        "missing_cells": df.isna().sum().sum(),
        "duplicate_rows": df.duplicated().sum(),
        "memory_usage_kb": round(
            df.memory_usage(deep=True).sum() / 1024, 2
        )
    })

dataset_profile = pd.DataFrame(profile_results)

display(dataset_profile)

print("\nDATASET PROFILING COMPLETED")
print("Total datasets:", len(portfolio_data))
print(
    "Total records:",
    dataset_profile["total_rows"].sum()
)

,table_name,total_rows,total_columns,missing_cells,duplicate_rows,memory_usage_kb
0,customer_details,4000,6,0,0,1359.01
1,loan_accounts,5000,8,0,0,1725.61
2,product_reference,5,2,0,0,0.43
3,repayment_schedule,224886,4,0,0,28769.72
4,payment_records,224886,10,11177,0,70885.99
5,account_summary,4178,4,0,0,534.62
6,portfolio_snapshots,230000,5,0,0,31220.83



DATASET PROFILING COMPLETED
Total datasets: 7
Total records: 692955


In [6]:
# ============================================================
# MILESTONE 1B: COLUMN-WISE MISSING VALUE ANALYSIS
# ============================================================

missing_results = []

for table_name, df in portfolio_data.items():

    for column in df.columns:

        missing_count = df[column].isna().sum()

        if missing_count > 0:

            missing_results.append({
                "table_name": table_name,
                "column_name": column,
                "missing_count": missing_count,
                "missing_percentage": round(
                    missing_count / len(df) * 100, 2
                )
            })

missing_profile = pd.DataFrame(missing_results)

if not missing_profile.empty:
    display(
        missing_profile.sort_values(
            "missing_count",
            ascending=False
        )
    )
else:
    print("No missing values detected.")

,table_name,column_name,missing_count,missing_percentage
0,payment_records,payment_date,11177,4.97


In [7]:
# ============================================================
# MILESTONE 1C: MISSING PAYMENT DATE INVESTIGATION
# ============================================================

payment_df = portfolio_data["payment_records"]

# Display all available columns
print("PAYMENT RECORD COLUMNS:")
print(payment_df.columns.tolist())

# Inspect records with missing payment dates
missing_payment_dates = payment_df[
    payment_df["payment_date"].isna()
]

print("\nMISSING PAYMENT DATE RECORDS:")
print("Total:", len(missing_payment_dates))

display(missing_payment_dates.head(15))

# Check whether missing dates are associated
# with particular payment statuses

PAYMENT RECORD COLUMNS:
['account_id', 'instalment_number', 'due_date', 'amount_due', 'payment_status', 'amount_paid', 'payment_date', 'instalment_outstanding', 'dpd', 'dpd_bucket']

MISSING PAYMENT DATE RECORDS:
Total: 11177


,account_id,instalment_number,due_date,amount_due,payment_status,amount_paid,payment_date,instalment_outstanding,dpd,dpd_bucket
1,L000001,2,2025-09-01,333311.21,Unpaid,0.0,NaN,333311.21,425,90+ DPD
11,L000002,6,2026-08-25,11573.21,Unpaid,0.0,NaN,11573.21,67,61-90 DPD
34,L000002,29,2028-07-25,11573.21,Unpaid,0.0,NaN,11573.21,0,Current
50,L000002,45,2029-11-25,11573.21,Unpaid,0.0,NaN,11573.21,0,Current
69,L000003,4,2025-04-23,39223.09,Unpaid,0.0,NaN,39223.09,556,90+ DPD
139,L000003,74,2031-02-23,39223.09,Unpaid,0.0,NaN,39223.09,0,Current
140,L000003,75,2031-03-23,39223.09,Unpaid,0.0,NaN,39223.09,0,Current
154,L000004,5,2024-11-06,12739.84,Unpaid,0.0,NaN,12739.84,724,90+ DPD
226,L000006,11,2027-08-05,77411.24,Unpaid,0.0,NaN,77411.24,0,Current
241,L000007,2,2026-02-09,49945.64,Unpaid,0.0,NaN,49945.64,264,90+ DPD


In [8]:
# ============================================================
# MILESTONE 1D: PAYMENT DATE BUSINESS LOGIC VALIDATION
# ============================================================

missing_payment_dates = payment_df[
    payment_df["payment_date"].isna()
].copy()

# Analyse missing payment dates by payment status
status_analysis = (
    missing_payment_dates
    .groupby("payment_status", dropna=False)
    .agg(
        affected_records=("account_id", "size"),
        total_amount_paid=("amount_paid", "sum"),
        maximum_amount_paid=("amount_paid", "max")
    )
    .reset_index()
)

display(status_analysis)

# Check whether any missing-date record has a payment amount
print("\nRECORDS WITH MISSING DATE BUT POSITIVE PAYMENT:")

positive_payment_count = (
    (missing_payment_dates["amount_paid"] > 0).sum()
)

print(positive_payment_count)

,payment_status,affected_records,total_amount_paid,maximum_amount_paid
0,Unpaid,11177,0.0,0.0



RECORDS WITH MISSING DATE BUT POSITIVE PAYMENT:
0


In [9]:
# ============================================================
# MILESTONE 1E: PAYMENT STATUS CONSISTENCY VALIDATION
# ============================================================

# 1. Analyse all payment statuses
display(
    payment_df.groupby("payment_status", dropna=False)
    .agg(
        total_records=("account_id", "size"),
        missing_payment_dates=("payment_date", lambda x: x.isna().sum()),
        total_amount_paid=("amount_paid", "sum")
    )
    .reset_index()
)

# 2. Identify potential inconsistencies
payment_exceptions = payment_df[
    (
        payment_df["payment_status"].str.lower().eq("paid")
        & payment_df["payment_date"].isna()
    )
    |
    (
        payment_df["payment_status"].str.lower().eq("unpaid")
        & (payment_df["amount_paid"] > 0)
    )
].copy()

print("\nPAYMENT STATUS EXCEPTIONS")
print("Total exceptions:", len(payment_exceptions))

display(payment_exceptions.head(10))

,payment_status,total_records,missing_payment_dates,total_amount_paid
0,Paid,191114,0,5.694299e+09
1,Partially Paid,22595,0,4.049665e+08
2,Unpaid,11177,11177,0.000000e+00



PAYMENT STATUS EXCEPTIONS
Total exceptions: 0


,account_id,instalment_number,due_date,amount_due,payment_status,amount_paid,payment_date,instalment_outstanding,dpd,dpd_bucket


In [10]:

# ============================================================
# MILESTONE 2: CONSOLIDATED DATA QUALITY VALIDATION ENGINE
# ============================================================

validation_results = []

def add_check(control_name, exception_count, severity="HIGH"):
    validation_results.append({
        "control_name": control_name,
        "exception_count": int(exception_count),
        "severity": severity,
        "status": "PASS" if exception_count == 0 else "FAIL"
    })


# ------------------------------------------------------------
# 1. MANDATORY FIELD VALIDATION
# ------------------------------------------------------------

mandatory_fields = {
    "customer_details": ["customer_id"],
    "loan_accounts": ["account_id", "customer_id", "product_id"],
    "product_reference": ["product_id", "interest_rate"],
    "repayment_schedule": ["account_id"],
    "payment_records": ["account_id"],
    "portfolio_snapshots": [
        "account_id", "reporting_date", "dpd", "overdue_exposure"
    ],
    "account_summary": ["account_id"]
}

for table_name, columns in mandatory_fields.items():
    df = portfolio_data[table_name]

    for column in columns:
        add_check(
            f"{table_name} - NULL {column}",
            df[column].isna().sum()
        )


# ------------------------------------------------------------
# 2. DUPLICATE KEY VALIDATION
# ------------------------------------------------------------

duplicate_keys = {
    "customer_details": ["customer_id"],
    "loan_accounts": ["account_id"],
    "product_reference": ["product_id"],
    "portfolio_snapshots": ["account_id", "reporting_date"]
}

for table_name, keys in duplicate_keys.items():
    df = portfolio_data[table_name]

    duplicate_count = df.duplicated(
        subset=keys, keep=False
    ).sum()

    add_check(
        f"{table_name} - Duplicate key",
        duplicate_count
    )


# ------------------------------------------------------------
# 3. REFERENTIAL INTEGRITY VALIDATION
# ------------------------------------------------------------

customers = set(portfolio_data["customer_details"]["customer_id"].dropna())
products = set(portfolio_data["product_reference"]["product_id"].dropna())
accounts = set(portfolio_data["loan_accounts"]["account_id"].dropna())

loans = portfolio_data["loan_accounts"]

add_check(
    "Loan accounts - Invalid customer",
    (~loans["customer_id"].isin(customers)).sum()
)

add_check(
    "Loan accounts - Invalid product",
    (~loans["product_id"].isin(products)).sum()
)

for table_name in ["payment_records", "repayment_schedule", "portfolio_snapshots"]:
    df = portfolio_data[table_name]

    add_check(
        f"{table_name} - Invalid account",
        (~df["account_id"].isin(accounts)).sum()
    )


# ------------------------------------------------------------
# 4. FINANCIAL AND DPD LOGIC VALIDATION
# ------------------------------------------------------------

snapshots = portfolio_data["portfolio_snapshots"]

add_check("Negative DPD", (snapshots["dpd"] < 0).sum())

add_check(
    "Negative overdue exposure",
    (snapshots["overdue_exposure"] < 0).sum()
)

add_check(
    "Current DPD with overdue exposure",
    ((snapshots["dpd"] == 0) &
     (snapshots["overdue_exposure"] > 0)).sum()
)

add_check(
    "Overdue DPD without overdue exposure",
    ((snapshots["dpd"] > 0) &
     (snapshots["overdue_exposure"] == 0)).sum()
)


# ------------------------------------------------------------
# 5. PAYMENT STATUS VALIDATION
# ------------------------------------------------------------

payments = portfolio_data["payment_records"]
payment_status = payments["payment_status"].str.lower()

add_check(
    "Paid/Partially Paid - Missing payment date",
    (
        payment_status.isin(["paid", "partially paid"])
        & payments["payment_date"].isna()
    ).sum()
)

add_check(
    "Unpaid - Positive amount paid",
    (
        payment_status.eq("unpaid")
        & (payments["amount_paid"] > 0)
    ).sum()
)


# ------------------------------------------------------------
# 6. CONSOLIDATED VALIDATION REPORT
# ------------------------------------------------------------

validation_report = pd.DataFrame(validation_results)

display(
    validation_report.sort_values(
        ["status", "severity", "control_name"],
        ascending=[True, True, True]
    ).reset_index(drop=True)
)

total_checks = len(validation_report)
failed_checks = (validation_report["status"] == "FAIL").sum()

print("\nCONSOLIDATED DATA QUALITY REPORT")
print("--------------------------------")
print("Total controls:", total_checks)
print("Passed controls:", total_checks - failed_checks)
print("Failed controls:", failed_checks)

print(
    "\nOVERALL STATUS:",
    "PASSED" if failed_checks == 0 else "EXCEPTIONS DETECTED"
)


,control_name,exception_count,severity,status
0,Current DPD with overdue exposure,0,HIGH,PASS
1,Loan accounts - Invalid customer,0,HIGH,PASS
2,Loan accounts - Invalid product,0,HIGH,PASS
3,Negative DPD,0,HIGH,PASS
4,Negative overdue exposure,0,HIGH,PASS
5,Overdue DPD without overdue exposure,0,HIGH,PASS
6,Paid/Partially Paid - Missing payment date,0,HIGH,PASS
7,Unpaid - Positive amount paid,0,HIGH,PASS
8,account_summary - NULL account_id,0,HIGH,PASS
9,customer_details - Duplicate key,0,HIGH,PASS



CONSOLIDATED DATA QUALITY REPORT
--------------------------------
Total controls: 28
Passed controls: 28
Failed controls: 0

OVERALL STATUS: PASSED


In [11]:

# ============================================================
# MILESTONE 3: EXCEPTION REPORT & SEVERITY CLASSIFICATION
# ============================================================

# Keep only failed controls
exception_report = validation_report[
    validation_report["status"] == "FAIL"
].copy()

# Assign severity based on control type
def classify_severity(control_name):
    critical_keywords = [
        "Invalid customer",
        "Invalid product",
        "Invalid account",
        "Duplicate key"
    ]

    if any(keyword in control_name for keyword in critical_keywords):
        return "CRITICAL"

    return "HIGH"


if not exception_report.empty:
    exception_report["severity"] = (
        exception_report["control_name"].apply(classify_severity)
    )

    display(exception_report)

else:
    print("No data quality exceptions detected.")

# Reporting gate
critical_count = (
    (exception_report["severity"] == "CRITICAL").sum()
    if not exception_report.empty else 0
)

print("\nREPORTING GATE")
print("----------------------------")
print("Critical exceptions:", critical_count)

reporting_status = (
    "BLOCKED" if critical_count > 0 else "ELIGIBLE FOR NEXT VALIDATION"
)

print("Reporting status:", reporting_status)


No data quality exceptions detected.

REPORTING GATE
----------------------------
Critical exceptions: 0
Reporting status: ELIGIBLE FOR NEXT VALIDATION


In [12]:

# ============================================================
# MILESTONE 4: CONTROLLED DEFECT INJECTION TEST
# ============================================================

# 1. Create a temporary copy of the loan accounts
test_loans = portfolio_data["loan_accounts"].copy()

# 2. Inject one duplicate account record
test_loans = pd.concat(
    [test_loans, test_loans.iloc[[0]]],
    ignore_index=True
)

# 3. Run the duplicate-key control
duplicate_count = test_loans.duplicated(
    subset=["account_id"],
    keep=False
).sum()

print("CONTROLLED DEFECT TEST")
print("----------------------")
print("Original loan records:", len(portfolio_data["loan_accounts"]))
print("Test loan records:", len(test_loans))
print("Duplicate records detected:", duplicate_count)

# 4. Simulate the reporting gate
test_status = "BLOCKED" if duplicate_count > 0 else "PASSED"

print("Reporting gate:", test_status)

# 5. Confirm the original data remains unchanged
print(
    "Original dataset unchanged:",
    len(portfolio_data["loan_accounts"]) == 5000
)


CONTROLLED DEFECT TEST
----------------------
Original loan records: 5000
Test loan records: 5001
Duplicate records detected: 2
Reporting gate: BLOCKED
Original dataset unchanged: True


In [13]:
# ============================================================
# PHASE 4: SAVE VALIDATION EVIDENCE
# ============================================================

reports_folder = project_folder / "reports"
reports_folder.mkdir(exist_ok=True)

validation_report.to_csv(
    reports_folder / "python_data_quality_validation.csv",
    index=False
)

exception_report.to_csv(
    reports_folder / "python_data_quality_exceptions.csv",
    index=False
)

print("Phase 4 evidence saved successfully.")
print("Location:", reports_folder)

Phase 4 evidence saved successfully.
Location: C:\DATA ANALYSIS\financial-risk-portfolio-analytics\reports


In [14]:

# ============================================================
# PHASE 5 - TASK 1: SQL-TO-PYTHON RECONCILIATION
# ============================================================

snapshots = portfolio_data["portfolio_snapshots"].copy()
summary = portfolio_data["account_summary"].copy()

# 1. Identify latest reporting date
latest_date = snapshots["reporting_date"].max()

# 2. Select latest overdue snapshot accounts
latest_overdue = snapshots[
    (snapshots["reporting_date"] == latest_date) &
    (snapshots["dpd"] > 0)
][
    ["account_id", "overdue_instalments", "overdue_exposure"]
].copy()

# 3. Full outer join: compare snapshot with account summary
recon = summary.merge(
    latest_overdue,
    on="account_id",
    how="outer",
    suffixes=("_summary", "_snapshot"),
    indicator=True
)

# 4. Identify missing records
missing_from_summary = (recon["_merge"] == "right_only").sum()
missing_from_snapshot = (recon["_merge"] == "left_only").sum()

# 5. Compare financial fields
matched = recon[recon["_merge"] == "both"].copy()

instalment_match = (
    pd.to_numeric(matched["overdue_instalments_summary"])
    == pd.to_numeric(matched["overdue_instalments_snapshot"])
)

exposure_match = np.isclose(
    pd.to_numeric(matched["overdue_exposure_summary"]),
    pd.to_numeric(matched["overdue_exposure_snapshot"]),
    atol=0.01,
    rtol=0,
    equal_nan=True
)

financial_mismatches = (~(instalment_match & exposure_match)).sum()
matched_accounts = len(matched) - financial_mismatches

# 6. Display reconciliation results
reconciliation_report = pd.DataFrame({
    "metric": [
        "Latest reporting date",
        "Missing from summary",
        "Missing from snapshot",
        "Financial mismatches",
        "Matched accounts"
    ],
    "result": [
        str(latest_date),
        int(missing_from_summary),
        int(missing_from_snapshot),
        int(financial_mismatches),
        int(matched_accounts)
    ]
})

display(reconciliation_report)


,metric,result
0,Latest reporting date,2026-10-31
1,Missing from summary,0
2,Missing from snapshot,0
3,Financial mismatches,0
4,Matched accounts,4178


In [16]:

# ============================================================
# PHASE 5 - TASK 2: PORTFOLIO KPI VALIDATION
# ============================================================

# STEP 1: LOAD LATEST PORTFOLIO SNAPSHOT
# ============================================================

snapshots = portfolio_data["portfolio_snapshots"].copy()

# Ensure reporting_date is in datetime format
snapshots["reporting_date"] = pd.to_datetime(
    snapshots["reporting_date"]
)

# Identify latest reporting date
latest_date = snapshots["reporting_date"].max()

# Filter latest portfolio
latest_portfolio = snapshots[
    snapshots["reporting_date"] == latest_date
].copy()

print("PORTFOLIO REPORTING DATE:", latest_date.date())
print("Total snapshot records:", len(latest_portfolio))


# STEP 2: CALCULATE PORTFOLIO KPIs
# ============================================================

total_accounts = latest_portfolio["account_id"].nunique()

overdue_accounts = latest_portfolio.loc[
    latest_portfolio["dpd"] > 0,
    "account_id"
].nunique()

current_accounts = latest_portfolio.loc[
    latest_portfolio["dpd"] == 0,
    "account_id"
].nunique()

# Overdue account percentage
overdue_account_rate = (
    overdue_accounts / total_accounts * 100
    if total_accounts > 0 else 0
)

# Total overdue exposure
total_overdue_exposure = latest_portfolio[
    "overdue_exposure"
].sum()

# Average overdue exposure per overdue snapshot record
average_overdue_exposure = latest_portfolio.loc[
    latest_portfolio["dpd"] > 0,
    "overdue_exposure"
].mean()

# Maximum DPD
maximum_dpd = latest_portfolio["dpd"].max()


# STEP 3: CREATE PORTFOLIO KPI REPORT
# ============================================================

portfolio_kpi_report = pd.DataFrame({
    "KPI": [
        "Reporting date",
        "Total accounts",
        "Overdue accounts",
        "Current accounts",
        "Overdue account rate (%)",
        "Total overdue exposure",
        "Average overdue exposure",
        "Maximum DPD"
    ],

    "Value": [
        latest_date.strftime("%Y-%m-%d"),
        total_accounts,
        overdue_accounts,
        current_accounts,
        round(overdue_account_rate, 2),
        round(total_overdue_exposure, 2),
        round(average_overdue_exposure, 2),
        maximum_dpd
    ]
})

print("\nPORTFOLIO KPI SUMMARY")
print("=" * 55)

display(portfolio_kpi_report)


# STEP 4: DPD BUCKET DISTRIBUTION
# ============================================================

dpd_buckets = {

    "1-30 DPD": (
        (latest_portfolio["dpd"] >= 1) &
        (latest_portfolio["dpd"] <= 30)
    ).sum(),

    "31-60 DPD": (
        (latest_portfolio["dpd"] >= 31) &
        (latest_portfolio["dpd"] <= 60)
    ).sum(),

    "61-89 DPD": (
        (latest_portfolio["dpd"] >= 61) &
        (latest_portfolio["dpd"] <= 89)
    ).sum(),

    "90+ DPD": (
        latest_portfolio["dpd"] >= 90
    ).sum()
}


# STEP 5: CREATE DPD BUCKET REPORT
# ============================================================

dpd_bucket_report = pd.DataFrame(
    list(dpd_buckets.items()),
    columns=[
        "DPD Bucket",
        "Account Count"
    ]
)

# Calculate percentage against total portfolio accounts
dpd_bucket_report["Account Percentage"] = (
    dpd_bucket_report["Account Count"] /
    total_accounts * 100
).round(2)


print("\nDPD BUCKET DISTRIBUTION")
print("=" * 55)

display(dpd_bucket_report)


# STEP 6: DPD BUCKET RECONCILIATION
# ============================================================

total_bucket_accounts = sum(dpd_buckets.values())

print("\nDPD BUCKET RECONCILIATION")
print("=" * 55)

print("Total bucket accounts:", total_bucket_accounts)
print("Total overdue accounts:", overdue_accounts)

print(
    "Bucket reconciliation:",
    "PASSED" if total_bucket_accounts == overdue_accounts
    else "FAILED"
)


# STEP 7: FINAL PORTFOLIO VALIDATION
# ============================================================

print("\nFINAL PORTFOLIO VALIDATION")
print("=" * 55)

print(
    "Account reconciliation:",
    "PASSED" if (
        total_accounts == current_accounts + overdue_accounts
    ) else "FAILED"
)

print(
    "DPD bucket reconciliation:",
    "PASSED" if (
        total_bucket_accounts == overdue_accounts
    ) else "FAILED"
)

print("\nPORTFOLIO KPI CALCULATION COMPLETED")


PORTFOLIO REPORTING DATE: 2026-10-31
Total snapshot records: 5000

PORTFOLIO KPI SUMMARY


,KPI,Value
0,Reporting date,2026-10-31
1,Total accounts,5000
2,Overdue accounts,4178
3,Current accounts,822
4,Overdue account rate (%),83.56
5,Total overdue exposure,341748689.09
6,Average overdue exposure,81797.2
7,Maximum DPD,1365



DPD BUCKET DISTRIBUTION


,DPD Bucket,Account Count,Account Percentage
0,1-30 DPD,215,4.30
1,31-60 DPD,103,2.06
2,61-89 DPD,96,1.92
3,90+ DPD,3764,75.28



DPD BUCKET RECONCILIATION
Total bucket accounts: 4178
Total overdue accounts: 4178
Bucket reconciliation: PASSED

FINAL PORTFOLIO VALIDATION
Account reconciliation: PASSED
DPD bucket reconciliation: PASSED

PORTFOLIO KPI CALCULATION COMPLETED


In [17]:

# ============================================================
# PHASE 5 - TASK 3: EXPORT PORTFOLIO REPORTING DATASETS
# ============================================================

reports_folder = project_folder / "reports"
reports_folder.mkdir(exist_ok=True)

# Export portfolio KPI summary
portfolio_kpi_report.to_csv(
    reports_folder / "portfolio_kpi_summary.csv",
    index=False
)

# Export DPD bucket distribution
dpd_bucket_report.to_csv(
    reports_folder / "dpd_bucket_distribution.csv",
    index=False
)

# Export reconciliation report if available
reconciliation_report.to_csv(
    reports_folder / "portfolio_reconciliation.csv",
    index=False
)

print("REPORTING DATASETS EXPORTED")
print("-" * 45)

for file_name in [
    "portfolio_kpi_summary.csv",
    "dpd_bucket_distribution.csv",
    "portfolio_reconciliation.csv"
]:
    file_path = reports_folder / file_name
    print(file_name, "—", "SAVED" if file_path.exists() else "NOT FOUND")


REPORTING DATASETS EXPORTED
---------------------------------------------
portfolio_kpi_summary.csv — SAVED
dpd_bucket_distribution.csv — SAVED
portfolio_reconciliation.csv — SAVED
